<a href="https://colab.research.google.com/github/armandoordonez/AI-Engineering/blob/main/RAG_con_trazabilidad_y_evidencia_visible.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# RAG con trazabilidad y evidencia visible

## Dataset seleccionado : FEVER

FEVER (Fact Extraction and VERification) es un benchmark diseñado exactamente para ese problema:

Contiene 185,445 afirmaciones creadas a partir de oraciones de Wikipedia.

Cada afirmación tiene una etiqueta:

- SUPPORTS
- REFUTES
- NOT ENOUGH INFO

Para las clases SUPPORTS y REFUTES, los anotadores también marcaron qué oración(es) de Wikipedia sirven como evidencia.

## ¿Qué y cómo vamos a medir?

- Recall@k: ¿Recuperé al menos una de las páginas correctas entre mis top-k?

- Precision@k: ¿Qué proporción de lo que recuperé era realmente relevante?

- Exactitud de clasificación: ¿Predije SUPPORTS/REFUTES/NEI correctamente?

- Cobertura de citas: ¿Pude asociar la afirmación a un fragmento recuperado?

## ¿Qué RAG estamos usando en el experimento?

No es un RAG de “pregunta-respuesta” clásico. Es un RAG de verificación de hechos:

- Input: una afirmación (claim).

- Corpus: Wikipedia (el corpus de FEVER).

- Recuperación:

  - BM25 (búsqueda léxica).

  - Dense retrieval (embeddings con sentence-transformers).

  - Híbrido (combinación de ambos).

- Verificación:

  - Similitud semántica entre claim y evidencia.

  - Opcionalmente un modelo NLI (cross-encoder/nli-deberta-v3-small) para inferir si la evidencia apoya o contradice el claim.

- Generación:

  - En modo template, no hay generación real; solo se construye un prompt.

  - En modo hf/openai/gemini/anthropic, un modelo genera una explicación en JSON.



**Claim → Recuperar evidencia en Wikipedia → Verificar con NLI/similitud → Decidir SUPPORTS/REFUTES/NEI → Mostrar el fragmento citado.**

# Pipeline de Verificación de Fact-Checking (FEVER)



## Paso 1: Preparación del Corpus

* 1.1. Carga de Datos:
  * Importar el corpus oficial de Wikipedia de FEVER en formato JSONL.
  * Cargar las etiquetas doradas (gold_labels) y la evidencia dorada (gold_evidence).
* 1.2. Segmentación (Chunking):
  * Dividir los artículos de Wikipedia a nivel de oración única (alineado con la evidencia de FEVER).
* 1.3. Estructuración del Metadata:
  * Asignar a cada fragmento el objeto estructurado:
    * chunk_id: Identificador único del fragmento.
    * page_id: Título/ID del artículo de Wikipedia.
    * text: Contenido textual de la oración.

## Paso 2: Construcción de Índices de Recuperación

* 2.1. Índice Léxico (BM25):
  * Tokenizar el corpus de chunks.
  * Indexar los fragmentos con rank_bm25 para la búsqueda por coincidencias clave.
* 2.2. Índice Denso (FAISS):
  * Generar embeddings para cada chunk usando el modelo sentence-transformers/all-MiniLM-L6-v2.
  * Indexar las representaciones vectoriales en una estructura faiss.IndexFlatIP o faiss.IndexHNSW.
* 2.3. Búsqueda Híbrida:
  * Implementar la combinación de métricas (p. ej., mediante Reciprocal Rank Fusion o normalización Z-score de puntuaciones BM25 y Dense).

## Paso 3: Pipeline de Verificación por Claim

Para cada afirmación (claim) del conjunto de datos:

* 3.1. Recuperación de Chunks (Retrieval):
  * Consultar el índice híbrido y extraer los top-k chunks más relevantes.
* 3.2. Evaluación del Retrieval (Métricas local-at-k):
  * recall_at_k: Proporción de páginas/oraciones doradas presentes en los top-k recuperados.
  * precision_at_k: Proporción de chunks recuperados que son verdaderamente relevantes.
  * hit_at_k: Indicador binario (1 si al menos una evidencia dorada está en los top-k; 0 en caso contrario).
* 3.3. Verificación Semántica e Inferencia de Lenguaje Natural (NLI):
  * Calcular la similitud semántica entre el claim y los chunks recuperados.
  * (Opcional) Evaluar el par (Claim, Chunk) en un modelo NLI para obtener probabilidades de:
    * entailment (apoyo)
    * contradiction (refutación)
    * neutral (no concluyente)
* 3.4. Asignación de Estado Interno:
  * soportada: Prevalece alta probabilidad de entailment.
  * contradictoria: Prevalece alta probabilidad de contradiction.
  * parcialmente soportada: Evidencia mixta o parcial.
  * no soportada: Predominio de estado neutral o evidencia insuficiente.
* 3.5. Mapeo a Etiquetas Oficiales FEVER:
  * soportada -> SUPPORTS
  * contradictoria -> REFUTES
  * resto -> NOT ENOUGH INFO (NEI)
* 3.6. Estimación de Confianza:
  * Calcular un score de confianza ponderado combinando la puntuación del retriever (BM25/Dense) y la certidumbre del modelo NLI.
* 3.7. Registro de Resultados (Output por Registro):
  * Guardar en un diccionario o DataFrame: claim, etiqueta_dorada, etiqueta_predicha, chunks_recuperados, chunk_de_soporte y métricas asociadas.

## Paso 4: Métricas de Evaluación Global

* 4.1. Evaluación de Clasificación:
  * Exactitud global (Accuracy): Porcentaje de etiquetas FEVER correctamente predichas (SUPPORTS, REFUTES, NEI).
  * Matriz de confusión e informe de clasificación (Precision, Recall, F1 por clase).
* 4.2. Métricas Promedio de Retrieval:
  * Mean Recall@k
  * Mean Precision@k
  * Mean Hit@k
* 4.3. Cobertura de Citación (Citation Coverage):
  * Tasa de claims a los que el sistema logró asignar al menos una cita/chunk de soporte válido.
* 4.4. Calibración de Confianza (Opcional):
  * Calcular Expected Calibration Error (ECE) para medir la alineación entre las probabilidades del modelo y la exactitud real.

# Marco teórico: verificación de afirmaciones con RAG sobre FEVER

En este experimento se implementa un sistema de **verificación de afirmaciones** (fact‑checking) basado en un pipeline de **RAG** (Retrieval‑Augmented Generation). El objetivo es decidir, para cada afirmación, si está **apoyada**, **refutada** o si hay **información insuficiente** según un corpus de referencia.

El conjunto de datos utilizado es **FEVER** (Fact Extraction and VERification), un benchmark estándar en esta tarea. [3][10]

## Dataset FEVER

**FEVER** es un dataset diseñado para evaluar sistemas automáticos de verificación de afirmaciones. [3]

Características principales:

- Contiene **185.445 afirmaciones** construidas a partir de oraciones de Wikipedia. [3]
- Cada afirmación tiene una etiqueta:
  - `SUPPORTS`: la evidencia en Wikipedia apoya la afirmación.
  - `REFUTES`: la evidencia contradice la afirmación.
  - `NOT ENOUGH INFO` (NEI): no hay evidencia suficiente para decidir. [3]
- Para las clases `SUPPORTS` y `REFUTES`, los anotadores marcaron qué **oraciones de Wikipedia** sirven como evidencia. [3]

En este notebook se utiliza el split de FEVER incluido en **BEIR**, un benchmark heterogéneo para recuperación de información. [2]

## Dataset FEVER

**FEVER** es un dataset diseñado para evaluar sistemas automáticos de verificación de afirmaciones. [3]

Características principales:

- Contiene **185.445 afirmaciones** construidas a partir de oraciones de Wikipedia. [3]
- Cada afirmación tiene una etiqueta:
  - `SUPPORTS`: la evidencia en Wikipedia apoya la afirmación.
  - `REFUTES`: la evidencia contradice la afirmación.
  - `NOT ENOUGH INFO` (NEI): no hay evidencia suficiente para decidir. [3]
- Para las clases `SUPPORTS` y `REFUTES`, los anotadores marcaron qué **oraciones de Wikipedia** sirven como evidencia. [3]

En este notebook se utiliza el split de FEVER incluido en **BEIR**, un benchmark heterogéneo para recuperación de información. [2]

## Tarea de verificación de afirmaciones

La tarea consiste en:

1. Recibir una **afirmación** (claim).
2. Recuperar fragmentos relevantes de un **corpus de Wikipedia**.
3. Usar esos fragmentos como **evidencia** para decidir si la afirmación es:
   - apoyada (`SUPPORTS`),
   - refutada (`REFUTES`), o
   - no concluyente (`NOT ENOUGH INFO`). [3][6]

A diferencia de un RAG clásico de pregunta‑respuesta, aquí no se genera una respuesta libre, sino que se **clasifica la relación** entre la afirmación y la evidencia recuperada. [6]

## Recuperación: BM25, densa e híbrida

Para encontrar evidencia relevante se usan tres estrategias:

### BM25 (recuperación léxica)

- **BM25** es un algoritmo de ranking basado en la frecuencia de términos y la longitud del documento. [2]
- Funciona bien para encontrar coincidencias de **nombres propios, términos técnicos y frases exactas**.
- No captura bien paráfrasis ni relaciones semánticas sin solapamiento léxico.

### Recuperación densa (embeddings)

- Se representa cada fragmento y cada afirmación como un **vector** usando un modelo de **sentence‑transformers**. [5]
- La similitud entre vectores (por ejemplo, producto punto o similitud coseno) indica qué tan **semánticamente cercanos** están.
- Permite recuperar textos relacionados aunque no compartan muchas palabras.

### Recuperación híbrida

- Combina las puntuaciones de BM25 y de los embeddings (por ejemplo, mediante fusión de rangos o normalización). [2]
- Aprovecha:
  - la precisión de BM25 para términos clave,
  - la capacidad semántica de los embeddings para paráfrasis.
- En benchmarks como **BEIR**, los métodos híbridos suelen mejorar métricas como **nDCG@10** y **Recall@k**. [2]

## Métricas de recuperación

Para evaluar qué tan bien recupera evidencia el sistema se usan métricas estándar de **Information Retrieval**. [2]

### Recall@k

- Mide la **proporción de documentos relevantes** que aparecen entre los primeros `k` resultados.
- Si hay una única evidencia relevante, coincide con la idea de “¿recuperé la evidencia correcta en el top‑k?”.

### Precision@k

- Mide la **proporción de los primeros `k` resultados** que son realmente relevantes.
- Penaliza cuando muchos de los primeros resultados son irrelevantes.

### nDCG@k (Normalized Discounted Cumulative Gain)

- Tiene en cuenta **cuántos documentos relevantes** hay y **en qué posición** aparecen.
- Los documentos relevantes más arriba contribuyen más al score.
- Es la métrica principal en BEIR para comparar sistemas de recuperación. [2]

### MRR@k (Mean Reciprocal Rank)

- Para cada consulta, toma el **inverso de la posición** del primer documento relevante.
- El promedio sobre todas las consultas da el MRR.
- Es útil cuando interesa especialmente que la **primera evidencia correcta** aparezca pronto. [2]

### Hit@k

- Métrica binaria por consulta: vale `1` si **al menos un documento relevante** está en el top‑`k`, y `0` en caso contrario.
- Promediando sobre todas las consultas se obtiene el Hit@k global.

## Verificación con similitud y NLI

Una vez recuperados los fragmentos, hay que decidir si apoyan o refutan la afirmación.

### Similitud semántica

- Se calcula la **similitud** entre el vector de la afirmación y el vector de cada fragmento.
- Una similitud alta sugiere que el fragmento habla de algo muy parecido, pero **no garantiza** que lo apoye o contradiga.

### Inferencia de lenguaje natural (NLI)

- Un modelo de **NLI** recibe como entrada:
  - **Premisa**: el fragmento recuperado.
  - **Hipótesis**: la afirmación.
- El modelo predice una de tres clases:
  - **entailment** (apoyo),
  - **contradiction** (refutación),
  - **neutral** (no concluyente). [6]
- En este experimento se usa un modelo tipo **cross‑encoder NLI** (por ejemplo, `nli-deberta-v3-small`), que evalúa la pareja completa y devuelve probabilidades para cada clase.

La decisión final (`SUPPORTS`, `REFUTES`, `NEI`) se obtiene combinando:

- la **puntuación de recuperación** (qué tan relevante parece el fragmento),
- la **salida del modelo NLI** (qué relación textual predice),
- y umbrales de confianza definidos en los parámetros.

## Métricas de clasificación y cobertura de citas

Además de evaluar la recuperación, se mide qué tan bien **clasifica** las afirmaciones.

### Exactitud de clasificación

- Proporción de afirmaciones cuya etiqueta predicha (`SUPPORTS`, `REFUTES`, `NEI`) coincide con la etiqueta dorada de FEVER. [3]
- También se pueden calcular **precisión, recall y F1** por clase mediante una matriz de confusión.

### Cobertura de citas (citation coverage)

- Mide en qué fracción de las afirmaciones el sistema logra:
  - recuperar al menos un fragmento relevante, y
  - asociar esa evidencia a la decisión tomada.
- Una cobertura alta indica que el sistema no solo clasifica, sino que **muestra evidencia** para justificarlo.

En sistemas de fact‑checking, es importante que la evidencia recuperada esté **alineada** con la decisión: no basta con clasificar bien; también hay que poder **explicar por qué**. [6]

## Resumen del pipeline

El flujo del experimento es:

1. **Preparar el corpus**:
   - Cargar artículos de Wikipedia del dataset FEVER.
   - Segmentar a nivel de **oración**, como en la evidencia original de FEVER. [3]
2. **Construir índices**:
   - Un índice **BM25** para búsqueda léxica.
   - Un índice **denso** con embeddings de `sentence-transformers`.
   - Una estrategia **híbrida** que combina ambos. [2]
3. **Para cada afirmación**:
   - Recuperar los `k` fragmentos más relevantes.
   - Calcular métricas de recuperación comparando con la evidencia dorada.
   - Usar similitud y/o NLI para decidir `SUPPORTS`, `REFUTES` o `NEI`.
   - Guardar la predicción, la confianza y el fragmento citado.
4. **Evaluar globalmente**:
   - Exactitud de clasificación.
   - Métricas promedio de recuperación (Recall@k, nDCG@k, MRR, Hit@k).
   - Cobertura de citas y calibración de confianza. [2][6]

## Implementación

### Instalación de dependencias

Esta celda instala las bibliotecas necesarias para:

- cargar datos (`datasets`, `pandas`),
- hacer cálculos numéricos (`numpy`, `torch`),
- construir índices BM25 y densos (`rank-bm25`, `faiss-cpu`, `sentence-transformers`),
- ejecutar modelos de NLI (`transformers`),
- visualizar resultados (`matplotlib`, `seaborn`).

El flag `-q` silencia la salida detallada de `pip`.

In [1]:
!pip -q install datasets pandas numpy scikit-learn sentence-transformers faiss-cpu rank-bm25 transformers accelerate sentencepiece matplotlib seaborn tqdm huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 48.9 MB/s eta 0:00:00


### Imports y configuración global

Esta celda:

- importa librerías estándar y de ML/IR,
- fija la semilla aleatoria (`SEED`) para reproducibilidad,
- selecciona el dispositivo (`cuda` o `cpu`),
- define los parámetros del experimento:
  - número de consultas, tamaño del corpus,
  - valores de `k` para recuperación y evaluación,
  - modo de retrieval (`bm25`, `dense`, `hybrid`),
  - modo de verificación (`similarity`, `nli`, `hybrid`),
  - umbrales de decisión y pesos de confianza.

In [2]:
import os, re, json, random, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

warnings.filterwarnings("ignore")
sns.set_theme(style="whitegrid")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# ---------------- Parámetros del experimento ----------------
QRELS_SPLIT   = "test"     # split de qrels de BEIR/FEVER
N_QUERIES     = 200        # claims a evaluar
N_DISTRACT    = 10_000     # páginas "de relleno" además de las doradas
CHUNK_SENTENCES = 1        # FEVER usa evidencia por oración

K             = 5          # chunks que se pasan al verificador
K_PAGES       = 10         # top-k páginas para métricas con qrels
K_CHUNKS      = 50         # chunks recuperados antes de agrupar por página
EVAL_KS       = list(range(1, 11))

RETRIEVAL_MODE = "hybrid"  # "bm25", "dense", "hybrid" (modo usado en la verificación)
HYBRID_ALPHA   = 0.55      # peso del denso en el híbrido
VERIFIER_MODE  = "hybrid"  # "similarity", "nli", "hybrid"
NLI_THRESHOLD  = 0.50      # prob. mínima para afirmar SUPPORTS/REFUTES
SIM_GATE       = 0.25      # en modo hybrid: score de retrieval mínimo para no devolver NEI
CONF_W_NLI     = 0.7       # peso del NLI en la confianza combinada

EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
NLI_MODEL       = "cross-encoder/nli-deberta-v3-small"

OUTPUT_DIR = Path("outputs_fever")
OUTPUT_DIR.mkdir(exist_ok=True)
print("Dispositivo:", DEVICE)

Dispositivo: cuda


### 1. Qrels y queries (BEIR / FEVER)

### Carga de qrels y queries (BEIR / FEVER)

Esta celda:

- descarga el split de **qrels** de FEVER desde BEIR,
- carga las relaciones `query-id` ↔ `corpus-id` con su score,
- muestra:
  - cuántos pares de relevancia hay,
  - cuántas consultas (afirmaciones) existen,
  - un ejemplo de los qrels y de los chunks del corpus.

Los qrels sirven como **verdad de referencia** para calcular Recall@k, nDCG@k, MRR, etc. [2]

In [3]:
from huggingface_hub import hf_hub_download, list_repo_files
from datasets import load_dataset

def read_qrels(split="test", repo="BeIR/fever-qrels"):
    # Devuelve un DataFrame [query-id, corpus-id, score] con score > 0.
    # Funciona con TSV con o sin cabecera (la cabecera era la causa del TypeError original).
    df = None
    try:
        files = list_repo_files(repo, repo_type="dataset")
        cand = [f for f in files
                if split in os.path.basename(f).lower()
                and f.lower().endswith((".tsv", ".csv", ".parquet"))]
        if cand:
            path = hf_hub_download(repo, cand[0], repo_type="dataset")
            print("Qrels descargados desde:", cand[0])
            if path.endswith(".parquet"):
                df = pd.read_parquet(path)
            else:
                df = pd.read_csv(path, sep="\t", header=None, dtype=str)
    except Exception as e:
        print("Descarga directa falló, intento con load_dataset ->", repr(e))
    if df is None:
        df = load_dataset(repo, split=split).to_pandas()

    df = df.iloc[:, :3].copy()
    df.columns = ["query-id", "corpus-id", "score"]
    df["score"] = pd.to_numeric(df["score"], errors="coerce")
    df = df.dropna(subset=["score"])          # descarta la fila de cabecera si venía como texto
    df = df[df["score"] > 0].copy()
    df["query-id"]  = df["query-id"].astype(str)
    df["corpus-id"] = df["corpus-id"].astype(str)
    return df.reset_index(drop=True)

qrels_df = read_qrels(QRELS_SPLIT)
print(f"Qrels ({QRELS_SPLIT}): {len(qrels_df):,} pares | {qrels_df['query-id'].nunique():,} queries")
display(qrels_df.head(3))

# Queries (claims) que SÍ tienen qrels
q_all = load_dataset("BeIR/fever", "queries", split="queries").to_pandas()
q_all["_id"] = q_all["_id"].astype(str)
q_pool = q_all[q_all["_id"].isin(set(qrels_df["query-id"]))]
assert len(q_pool) > 0, "Ningún id de queries coincide con los qrels: revisa QRELS_SPLIT / formato de ids"

q_eval = (q_pool.sample(n=min(N_QUERIES, len(q_pool)), random_state=SEED)
                .reset_index(drop=True)
                .rename(columns={"_id": "id"}))
q_eval["id"] = q_eval["id"].astype(str)

qrels_full = {}
for r in qrels_df[qrels_df["query-id"].isin(set(q_eval["id"]))].itertuples(index=False):
    qrels_full.setdefault(r[0], {})[r[1]] = int(r[2])
gold_ids = {d for rel in qrels_full.values() for d in rel}
print(f"Queries seleccionadas: {len(q_eval)} | páginas doradas distintas: {len(gold_ids)}")

test.tsv:   0%|          | 0.00/210k [00:00<?, ?B/s]

Qrels descargados desde: test.tsv
Qrels (test): 7,937 pares | 6,666 queries


,query-id,corpus-id,score
0,163803,Ukrainian_Soviet_Socialist_Republic,1.0
1,163803,United_Nations,1.0
2,70041,2_Hearts_(Kylie_Minogue_song),1.0


README.md:   0%|          | 0.00/10.4k [00:00<?, ?B/s]

queries/queries-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 5.04MB            

queries/queries-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating queries split:   0%|          | 0/123142 [00:00<?, ? examples/s]

Queries seleccionadas: 200 | páginas doradas distintas: 208


### Construcción del corpus y chunking

Esta celda:

- carga los documentos de Wikipedia asociados a FEVER,
- segmenta cada artículo en **oraciones** (chunks),
- asigna a cada chunk:
  - un `chunk_id` único,
  - el `page_id` (título de la página),
  - el texto de la oración.

El chunking a nivel de oración está alineado con la evidencia anotada en FEVER. [3]

In [ ]:
# Se recorre el corpus en streaming hasta encontrar TODAS las páginas doradas
# y reunir N_DISTRACT distractores. El resultado se guarda en caché (parquet).
CACHE = OUTPUT_DIR / f"corpus_subset_q{len(q_eval)}_d{N_DISTRACT}_s{SEED}.parquet"

if CACHE.exists():
    print("Usando caché:", CACHE)
    raw_df = pd.read_parquet(CACHE)
else:
    corpus_stream = load_dataset("BeIR/fever", "corpus", split="corpus", streaming=True)
    rows, found, n_dis = [], set(), 0
    for row in tqdm(corpus_stream, desc="Recorriendo corpus"):
        rid = str(row.get("_id", row.get("id")))      # la columna real es "_id"
        if rid in gold_ids:
            if rid not in found:
                rows.append({"page_id": rid, "title": row.get("title"), "text": row.get("text")})
                found.add(rid)
        elif n_dis < N_DISTRACT:
            rows.append({"page_id": rid, "title": row.get("title"), "text": row.get("text")})
            n_dis += 1
        if len(found) == len(gold_ids) and n_dis >= N_DISTRACT:
            break
    raw_df = pd.DataFrame(rows)
    raw_df.to_parquet(CACHE)

wiki_df = raw_df.copy()
wiki_df["page_id"] = wiki_df["page_id"].astype(str)
wiki_df["title"]   = wiki_df["title"].fillna("").astype(str)
wiki_df["text"]    = (wiki_df["title"] + ". " + wiki_df["text"].fillna("").astype(str)).str.strip()
wiki_df = wiki_df.drop_duplicates("page_id")
wiki_df = wiki_df[wiki_df["text"].str.len() > 0].reset_index(drop=True)

# Qrels efectivos: solo páginas que realmente están en el corpus
corpus_ids = set(wiki_df["page_id"])
qrels = {q: {d: s for d, s in rel.items() if d in corpus_ids} for q, rel in qrels_full.items()}
qrels = {q: rel for q, rel in qrels.items() if rel}
n_lost = len(gold_ids - corpus_ids)
print(f"Documentos en corpus: {len(wiki_df):,}")
print(f"Páginas doradas encontradas: {len(gold_ids) - n_lost}/{len(gold_ids)}"
      f"  |  queries con qrels utilizables: {len(qrels)}/{len(q_eval)}")

q_eval = q_eval[q_eval["id"].isin(qrels)].reset_index(drop=True)
avg_rel = np.mean([len(v) for v in qrels.values()])
print(f"Páginas relevantes por query (promedio): {avg_rel:.2f}")
ex = q_eval.iloc[0]
print("\nEjemplo ->", ex["text"], "\n   qrels:", qrels[ex["id"]])

Recorriendo corpus: 0it [00:00, ?it/s]

### 3. Chunking por oración

In [ ]:
def split_sentences(text):
    text = re.sub(r"\s+", " ", str(text)).strip()
    if not text:
        return []
    return [s.strip() for s in re.split(r"(?<=[.!?])\s+", text) if s.strip()]

def build_chunks(df, sentences_per_chunk=1):
    rows = []
    for page_id, title, text in tqdm(zip(df["page_id"], df["title"], df["text"]),
                                     total=len(df), desc="Fragmentando"):
        sents = split_sentences(text)
        for start in range(0, len(sents), sentences_per_chunk):
            part = sents[start:start + sentences_per_chunk]
            if part:
                rows.append({
                    "chunk_id": f"{page_id}_s{start}",
                    "page_id": page_id,
                    "title": title,
                    "sentence_start": start,
                    "sentence_end": start + len(part) - 1,
                    "text": " ".join(part),
                })
    return pd.DataFrame(rows)

chunks = build_chunks(wiki_df, CHUNK_SENTENCES)
page_arr = chunks["page_id"].to_numpy()
print("Fragmentos creados:", len(chunks))
display(chunks.head(3))

### Construcción del índice BM25

Esta celda:

- tokeniza los chunks del corpus,
- crea un índice **BM25** con `rank-bm25`,
- define una función que, dada una consulta, devuelve los `k` chunks con mayor puntuación BM25.

BM25 usa frecuencia de términos y longitud del documento para rankear. [2]

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
import faiss

def tokenize(t):
    return re.findall(r"\w+", t.lower())

bm25 = BM25Okapi([tokenize(t) for t in chunks["text"]])

embedder = SentenceTransformer(EMBEDDING_MODEL, device=DEVICE)
chunk_embeddings = embedder.encode(
    chunks["text"].tolist(), normalize_embeddings=True, convert_to_numpy=True,
    show_progress_bar=True, batch_size=128
).astype("float32")

faiss_index = faiss.IndexFlatIP(chunk_embeddings.shape[1])
faiss_index.add(chunk_embeddings)
print("Índices listos | chunks:", len(chunks), "| dim:", chunk_embeddings.shape[1])

### Embeddings y índice denso (FAISS)

Esta celda:

- carga un modelo de **sentence‑transformers**,
- calcula el embedding de cada chunk,
- construye un índice **FAISS** (por ejemplo, `IndexFlatIP`),
- define una función que, dada la embedding de una consulta, devuelve los `k` chunks más similares.

FAISS permite búsqueda eficiente de vecinos cercanos en espacios de alta dimensión. [5]

In [ ]:
def minmax(values):
    values = np.asarray(values, dtype=np.float32)
    rng = values.max() - values.min()
    if rng < 1e-12:
        return np.zeros_like(values)
    return (values - values.min()) / rng

def score_all(query):
    # Scores de TODOS los chunks, alineados por posición con `chunks`.
    sparse = np.asarray(bm25.get_scores(tokenize(query)), dtype=np.float32)

    q_emb = embedder.encode([query], normalize_embeddings=True,
                            convert_to_numpy=True).astype("float32")
    D, I = faiss_index.search(q_emb, len(chunks))
    # CORRECCIÓN: FAISS devuelve D ordenado por ranking. Hay que reubicar cada score
    # en la posición de su chunk (I) antes de combinarlo con BM25.
    dense = np.zeros(len(chunks), dtype=np.float32)
    dense[I[0]] = D[0]
    return minmax(sparse), minmax(dense)

def combine(sparse, dense, mode, alpha=HYBRID_ALPHA):
    if mode == "bm25":   return sparse
    if mode == "dense":  return dense
    if mode == "hybrid": return alpha * dense + (1 - alpha) * sparse
    raise ValueError("mode debe ser 'bm25', 'dense' o 'hybrid'")

def aggregate_pages(scores, k_chunks=K_CHUNKS, k_pages=K_PAGES):
    # Top chunks -> páginas (score de página = mejor chunk). Devuelve {page_id: score} ordenado.
    k_chunks = min(k_chunks, len(scores))
    idx = np.argpartition(-scores, k_chunks - 1)[:k_chunks]
    idx = idx[np.argsort(-scores[idx])]
    best = {}
    for i in idx:
        p = page_arr[i]
        if p not in best:
            best[p] = float(scores[i])
            if len(best) >= k_pages:
                break
    return best

def top_chunks(sparse, dense, mode=RETRIEVAL_MODE, k=K):
    scores = combine(sparse, dense, mode)
    idx = np.argsort(-scores)[:k]
    out = chunks.iloc[idx].copy().reset_index(drop=True)
    out["sparse_score"] = sparse[idx]
    out["dense_score"] = dense[idx]
    out["retrieval_score"] = scores[idx]
    return out

# Se calculan una sola vez los scores completos por query (reutilizados por todos los modos)
sparse_cache, dense_cache = {}, {}
for qid, claim in tqdm(zip(q_eval["id"], q_eval["text"]), total=len(q_eval), desc="Scoring"):
    sparse_cache[qid], dense_cache[qid] = score_all(claim)

def build_run(mode, alpha=HYBRID_ALPHA):
    return {qid: aggregate_pages(combine(sparse_cache[qid], dense_cache[qid], mode, alpha))
            for qid in q_eval["id"]}

runs = {m: build_run(m) for m in ["bm25", "dense", "hybrid"]}
print({m: len(r) for m, r in runs.items()}, "queries rankeadas")

### Retrieval híbrido (BM25 + denso)

Esta celda implementa la estrategia híbrida combinando las puntuaciones de BM25 y del índice denso: normaliza ambas escalas, las pondera con un parámetro `alpha` (por ejemplo, 0.55 para dar más peso al componente denso) y rankea los chunks según la suma ponderada. De esta forma, el sistema aprovecha tanto las coincidencias léxicas fuertes de BM25 como la capacidad de los embeddings para capturar paráfrasis y relaciones semánticas. [2]

In [ ]:
def evaluate_run(qrels, run, ks):
    # Métricas por query con relevancia binaria (score>0). Devuelve DataFrame indexado por qid.
    # Las queries sin resultados cuentan como 0 (no se omiten).
    rows = {}
    for qid, rel in qrels.items():
        ranked = sorted(run.get(qid, {}).items(), key=lambda x: -x[1])
        pages = [p for p, _ in ranked]
        rels = {d for d, s in rel.items() if s > 0}
        row = {}
        for k in ks:
            hits = [p in rels for p in pages[:k]]
            n_hit = sum(hits)
            dcg  = sum(1.0 / np.log2(i + 2) for i, h in enumerate(hits) if h)
            idcg = sum(1.0 / np.log2(i + 2) for i in range(min(len(rels), k)))
            rr = next((1.0 / (i + 1) for i, h in enumerate(hits) if h), 0.0)
            row[f"Recall@{k}"]    = n_hit / len(rels) if rels else 0.0
            row[f"Precision@{k}"] = n_hit / k
            row[f"Hit@{k}"]       = float(n_hit > 0)
            row[f"nDCG@{k}"]      = dcg / idcg if idcg > 0 else 0.0
            row[f"MRR@{k}"]       = rr
        row["first_rel_rank"] = next((i + 1 for i, p in enumerate(pages) if p in rels), np.nan)
        rows[qid] = row
    return pd.DataFrame.from_dict(rows, orient="index")

# ---- Prueba unitaria de las métricas (valores calculados a mano) ----
_toy_q = {"q1": {"A": 1, "B": 1}}
_toy_r = {"q1": {"X": 4.0, "A": 3.0, "Y": 2.0, "B": 1.0}}
_m = evaluate_run(_toy_q, _toy_r, [2, 4]).iloc[0]
assert abs(_m["Recall@2"] - 0.5) < 1e-9 and abs(_m["Precision@2"] - 0.5) < 1e-9
assert _m["Hit@2"] == 1.0 and abs(_m["MRR@2"] - 0.5) < 1e-9
assert abs(_m["nDCG@2"] - (1/np.log2(3)) / (1 + 1/np.log2(3))) < 1e-9
assert abs(_m["Recall@4"] - 1.0) < 1e-9 and _m["first_rel_rank"] == 2
print("✓ Pruebas unitarias de métricas OK")

# ---- Comprobación de ids: el error silencioso típico es que run y qrels no compartan ids ----
for m, run in runs.items():
    assert set(run) == set(qrels), f"Los ids de la run '{m}' no coinciden con los qrels"
sample_qid = next(iter(qrels))
print("Ids de qrels vs ids recuperados (ejemplo):",
      list(qrels[sample_qid]), "<->", list(runs["hybrid"][sample_qid])[:3])

per_query = {m: evaluate_run(qrels, run, EVAL_KS) for m, run in runs.items()}
mean_metrics = pd.DataFrame({m: df.drop(columns="first_rel_rank").mean() for m, df in per_query.items()}).T
mean_metrics.index.name = "modo"

show_cols = ["nDCG@1", "nDCG@5", "nDCG@10", "Recall@1", "Recall@5", "Recall@10",
             "Precision@5", "Precision@10", "MRR@10", "Hit@1", "Hit@5", "Hit@10"]
print(f"\n=== MÉTRICAS DE RETRIEVAL A NIVEL DE PÁGINA ({len(qrels)} queries) ===")
display(mean_metrics[show_cols].round(4))
mean_metrics.to_csv(OUTPUT_DIR / "retrieval_metrics.csv")

#### Gráficas de retrieval

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
palette = {"bm25": "#4C72B0", "dense": "#DD8452", "hybrid": "#55A868"}

# (1) Comparación de modos
ax = axes[0, 0]
cols = ["nDCG@10", "Recall@10", "Precision@10", "MRR@10", "Hit@10"]
plot_df = mean_metrics[cols].reset_index().melt(id_vars="modo", var_name="métrica", value_name="valor")
sns.barplot(data=plot_df, x="métrica", y="valor", hue="modo", palette=palette, ax=ax)
for c in ax.containers:
    ax.bar_label(c, fmt="%.2f", fontsize=8)
ax.set_title("Retrieval vs qrels: BM25 / Denso / Híbrido (k=10)")
ax.set_ylim(0, 1.05); ax.set_xlabel("")

# (2) Curva Recall@k
ax = axes[0, 1]
for m in mean_metrics.index:
    ax.plot(EVAL_KS, [mean_metrics.loc[m, f"Recall@{k}"] for k in EVAL_KS],
            marker="o", label=m, color=palette[m])
ax.set_title("Recall@k de páginas"); ax.set_xlabel("k"); ax.set_ylabel("Recall@k")
ax.set_xticks(EVAL_KS); ax.set_ylim(0, 1.05); ax.legend()

# (3) Posición de la primera página relevante (híbrido)
ax = axes[1, 0]
fr = per_query["hybrid"]["first_rel_rank"]
counts = [int((fr == r).sum()) for r in range(1, K_PAGES + 1)] + [int(fr.isna().sum())]
labels = [str(r) for r in range(1, K_PAGES + 1)] + [f">{K_PAGES}"]
bars = ax.bar(labels, counts, color=["#55A868"] * K_PAGES + ["#C44E52"])
ax.bar_label(bars, fontsize=8)
ax.set_title("Posición de la 1ª página relevante (híbrido)")
ax.set_xlabel("Ranking"); ax.set_ylabel("# queries")

# (4) Barrido de alpha del híbrido
ax = axes[1, 1]
alphas = np.round(np.linspace(0, 1, 11), 2)
sweep = []
for a in alphas:
    ev = evaluate_run(qrels, build_run("hybrid", a), [10])
    sweep.append((ev["nDCG@10"].mean(), ev["Recall@10"].mean()))
sweep = np.array(sweep)
ax.plot(alphas, sweep[:, 0], marker="o", label="nDCG@10")
ax.plot(alphas, sweep[:, 1], marker="s", label="Recall@10")
ax.axvline(HYBRID_ALPHA, ls="--", color="gray", label=f"alpha actual = {HYBRID_ALPHA}")
best_a = alphas[sweep[:, 0].argmax()]
ax.set_title(f"Barrido de alpha (0 = BM25, 1 = denso) | mejor nDCG@10 en {best_a}")
ax.set_xlabel("alpha"); ax.set_ylabel("métrica"); ax.set_ylim(0, 1.05); ax.legend()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / "retrieval_metrics.png", dpi=150)
plt.show()

### Verificación: similitud y NLI

Esta celda toma cada afirmación y sus chunks recuperados, calcula la similitud semántica entre el claim y cada chunk, y (en modo NLI o híbrido) pasa las parejas `(claim, chunk)` por un modelo de inferencia de lenguaje natural (por ejemplo, `cross-encoder/nli-deberta-v3-small`) para obtener probabilidades de entailment, contradiction y neutral. Con estas señales y unos umbrales configurables, asigna a cada claim una etiqueta interna (`soportada`, `contradictoria`, `parcialmente soportada`, `no soportada`) y luego la mapea a las etiquetas oficiales de FEVER (`SUPPORTS`, `REFUTES`, `NEI`), además de calcular un score de confianza combinado. [6]

In [ ]:
from sentence_transformers import CrossEncoder

nli_model = CrossEncoder(NLI_MODEL, device=DEVICE)

# Orden real de las columnas del modelo (según su config). Para este modelo:
# 0 = contradiction, 1 = entailment, 2 = neutral
_default = {0: "contradiction", 1: "entailment", 2: "neutral"}
try:
    id2label = {int(k): str(v).lower() for k, v in nli_model.model.config.id2label.items()}
    if set(id2label.values()) != set(_default.values()):
        id2label = _default
except Exception:
    id2label = _default
col_of = {name: i for i, name in id2label.items()}
print("Orden de etiquetas NLI:", id2label)

def _softmax(x):
    e = np.exp(x - x.max(axis=1, keepdims=True))
    return e / e.sum(axis=1, keepdims=True)

def nli_probs(claim, evidence_chunks):
    # Probabilidades (contradiction, entailment, neutral) para cada chunk.
    # Premisa = evidencia, hipótesis = claim (dirección correcta en NLI).
    premises = []
    for title, text in zip(evidence_chunks["title"], evidence_chunks["text"]):
        premises.append(text if text.lower().startswith(title.lower()) else f"{title}: {text}")
    out = np.asarray(nli_model.predict([(p, claim) for p in premises], show_progress_bar=False), dtype=float)
    if out.ndim == 1:
        out = out[None, :]
    # Si ya vienen como probabilidades (suman 1) no se vuelve a aplicar softmax
    if not (np.all(out >= 0) and np.allclose(out.sum(axis=1), 1.0, atol=1e-3)):
        out = _softmax(out)
    return pd.DataFrame({
        "contradiction": out[:, col_of["contradiction"]],
        "entailment":    out[:, col_of["entailment"]],
        "neutral":       out[:, col_of["neutral"]],
    })

def classify_claim(retrieved, probs):
    # Devuelve (etiqueta FEVER, estado interno, confianza, posición del chunk citado)
    top_ret = float(retrieved["retrieval_score"].max())

    if VERIFIER_MODE == "similarity" or probs is None:
        cited = int(retrieved["retrieval_score"].values.argmax())
        if top_ret >= 0.6:
            return "SUPPORTS", "soportada", top_ret, cited
        return "NOT ENOUGH INFO", "no soportada", top_ret, cited

    sup, ref = probs["entailment"].values, probs["contradiction"].values
    i_sup, i_ref = int(sup.argmax()), int(ref.argmax())
    s, r = float(sup[i_sup]), float(ref[i_ref])

    if VERIFIER_MODE == "hybrid" and top_ret < SIM_GATE:      # evidencia recuperada muy débil
        return "NOT ENOUGH INFO", "no soportada", top_ret, i_sup

    if max(s, r) < NLI_THRESHOLD:
        return "NOT ENOUGH INFO", "no soportada", float(1 - max(s, r)), i_sup
    if s >= NLI_THRESHOLD and r >= NLI_THRESHOLD:
        state = "parcialmente soportada"                        # evidencia mixta
    else:
        state = "soportada" if s >= r else "contradictoria"
    if s >= r:
        label, i, p = "SUPPORTS", i_sup, s
    else:
        label, i, p = "REFUTES", i_ref, r
    conf = CONF_W_NLI * p + (1 - CONF_W_NLI) * float(retrieved["retrieval_score"].iloc[i])
    return label, state, float(conf), i

### Evaluación por claim (retrieval + clasificación)

Esta celda itera sobre las afirmaciones seleccionadas, compara los chunks recuperados con la evidencia dorada de FEVER y calcula, para cada claim, métricas de retrieval como Recall@k, Precision@k y Hit@k, indicando cuántas de las páginas/oraciones doradas aparecieron en el top‑k. Simultáneamente guarda la etiqueta dorada, la etiqueta predicha por el verificador, los chunks recuperados, el chunk de soporte principal y las métricas asociadas, construyendo así una tabla de resultados por registro que luego se agregará a nivel global. [2][3]

In [ ]:
records = []
for qid, claim in tqdm(zip(q_eval["id"], q_eval["text"]), total=len(q_eval), desc="Verificando claims"):
    retrieved = top_chunks(sparse_cache[qid], dense_cache[qid], RETRIEVAL_MODE, K)
    probs = nli_probs(claim, retrieved) if VERIFIER_MODE in ("nli", "hybrid") else None
    label, state, conf, ci = classify_claim(retrieved, probs)
    cited = retrieved.iloc[ci]
    records.append({
        "qid": qid, "claim": claim,
        "pred_label": label, "estado": state, "confidence": conf,
        "cited_chunk": cited["chunk_id"], "cited_page": cited["page_id"],
        "cited_text": cited["text"],
        "cited_in_qrels": cited["page_id"] in qrels[qid],
        "p_entail":  float(probs["entailment"].iloc[ci])    if probs is not None else np.nan,
        "p_contra":  float(probs["contradiction"].iloc[ci]) if probs is not None else np.nan,
    })
verif_df = pd.DataFrame(records)
verif_df.to_csv(OUTPUT_DIR / "verificacion_por_claim.csv", index=False)

print("Distribución de etiquetas predichas:")
print(verif_df["pred_label"].value_counts().to_string())
display(verif_df[["claim", "pred_label", "confidence", "cited_page", "cited_in_qrels"]].head(8))

### 9. Cobertura de citas y evidencia visible (validada con qrels)

In [ ]:
has_cite  = verif_df["pred_label"] != "NOT ENOUGH INFO"
coverage  = has_cite.mean()
cite_ok   = verif_df.loc[has_cite, "cited_in_qrels"].mean() if has_cite.any() else np.nan
cite_ok_all = verif_df["cited_in_qrels"].mean()

print(f"Cobertura de citas (claims con SUPPORTS/REFUTES y chunk citado): {coverage:.3f}")
print(f"Precisión de la cita vs qrels (página citada ∈ qrels, entre claims con cita): {cite_ok:.3f}")
print(f"Chunk top-1 del verificador cae en página dorada (todos los claims):          {cite_ok_all:.3f}")

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].bar(["Cobertura\nde citas", "Cita correcta\n(∈ qrels)", "Top-1 ∈ qrels\n(todos)"],
            [coverage, cite_ok if cite_ok == cite_ok else 0, cite_ok_all],
            color=["#4C72B0", "#55A868", "#DD8452"])
for i, v in enumerate([coverage, cite_ok if cite_ok == cite_ok else 0, cite_ok_all]):
    axes[0].text(i, v + 0.02, f"{v:.2f}", ha="center")
axes[0].set_ylim(0, 1.1); axes[0].set_title("Trazabilidad de la evidencia")

verif_df["estado"].value_counts().plot.barh(ax=axes[1], color="#8172B3")
axes[1].set_title("Estado interno asignado"); axes[1].set_xlabel("# claims")
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "citas.png", dpi=150)
plt.show()

print("\n--- Ejemplos con evidencia visible ---")
for _, r in verif_df.sample(min(5, len(verif_df)), random_state=SEED).iterrows():
    print(f"CLAIM     : {r['claim']}")
    print(f"PREDICCIÓN: {r['pred_label']} ({r['estado']}, conf={r['confidence']:.2f})")
    print(f"EVIDENCIA : [{r['cited_page']}] {r['cited_text'][:200]}")
    print(f"¿Página en qrels? {'sí' if r['cited_in_qrels'] else 'no'}\n")

### Métricas globales de evaluación

Esta celda agrega los resultados individuales para obtener métricas promedio de retrieval (Mean Recall@k, Mean Precision@k, Mean Hit@k, nDCG@k, MRR) y métricas de clasificación (exactitud global, matriz de confusión, precisión, recall y F1 por clase). También calcula la cobertura de citas (fracción de claims con al menos una cita válida asociada) y, opcionalmente, evalúa la calibración entre los scores de confianza y la exactitud real, permitiendo comparar el desempeño de los modos `bm25`, `dense` y `hybrid`. [2]

In [ ]:
import requests
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score

FEVER_URLS = [
    "https://fever.ai/download/fever/shared_task_dev.jsonl",
    "https://s3-eu-west-1.amazonaws.com/fever.public/shared_task_dev.jsonl",
]
FEVER_PATH = OUTPUT_DIR / "fever_dev.jsonl"

def load_fever_dev():
    if not FEVER_PATH.exists():
        for url in FEVER_URLS:
            try:
                resp = requests.get(url, timeout=120)
                resp.raise_for_status()
                FEVER_PATH.write_bytes(resp.content)
                print("✓ Descargado:", url)
                break
            except Exception as e:
                print("No se pudo descargar", url, "->", repr(e)[:100])
    if not FEVER_PATH.exists():
        return None
    with open(FEVER_PATH, encoding="utf-8") as f:
        return pd.DataFrame([json.loads(l) for l in f if l.strip()])

def norm_claim(s):
    return re.sub(r"\s+", " ", str(s)).strip().lower()

LABELS = ["SUPPORTS", "REFUTES", "NOT ENOUGH INFO"]
fever_df = load_fever_dev()
merged = pd.DataFrame()
if fever_df is not None:
    fever_df["key"] = fever_df["claim"].map(norm_claim)
    fever_df = fever_df.drop_duplicates("key")
    tmp = verif_df.copy()
    tmp["key"] = tmp["claim"].map(norm_claim)
    merged = tmp.merge(fever_df[["key", "label"]].rename(columns={"label": "gold_label"}), on="key", how="inner")
    merged["gold_label"] = merged["gold_label"].str.upper()
    print(f"Claims emparejados con etiqueta dorada: {len(merged)}/{len(verif_df)}")
else:
    print(" No hay etiquetas doradas; se omite la evaluación de clasificación (el retrieval con qrels sí se evaluó).")

clf_results = {}
if len(merged) > 0:
    print("\nDistribución de etiquetas doradas en la muestra:")
    print(merged["gold_label"].value_counts().to_string())

    y_true, y_pred = merged["gold_label"], merged["pred_label"]
    acc = accuracy_score(y_true, y_pred)
    maj = y_true.value_counts(normalize=True).max()
    macro_f1 = f1_score(y_true, y_pred, labels=LABELS, average="macro", zero_division=0)
    print(f"\n✓ ACCURACY: {acc:.4f}   |   Macro-F1: {macro_f1:.4f}   |   baseline clase mayoritaria: {maj:.4f}")
    print("\n" + classification_report(y_true, y_pred, labels=LABELS, digits=4, zero_division=0))
    clf_results = {"accuracy": acc, "macro_f1": macro_f1, "majority_baseline": maj, "n": len(merged)}
    merged.to_csv(OUTPUT_DIR / "resultados_con_gold.csv", index=False)

### Visualización de resultados

Esta celda genera gráficas con las métricas de retrieval para cada modo de búsqueda, muestra la matriz de confusión de la clasificación y presenta ejemplos concretos de claims con su etiqueta dorada, predicción, evidencia citada y score de confianza. Estas visualizaciones ayudan a interpretar en qué tipos de afirmaciones el sistema acierta o falla, y cómo se relacionan la calidad del retrieval y la precisión de la verificación.

In [ ]:
if len(merged) > 0:
    cm = confusion_matrix(merged["gold_label"], merged["pred_label"], labels=LABELS)
    cm_df = pd.DataFrame(cm, index=LABELS, columns=LABELS)

    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8))
    sns.heatmap(cm_df, annot=True, fmt="d", cmap="Blues", cbar=False, ax=axes[0])
    axes[0].set_title("Matriz de confusión"); axes[0].set_ylabel("Dorado"); axes[0].set_xlabel("Predicho")

    dist = pd.DataFrame({
        "dorado":   merged["gold_label"].value_counts().reindex(LABELS).fillna(0),
        "predicho": merged["pred_label"].value_counts().reindex(LABELS).fillna(0),
    })
    dist.plot.bar(ax=axes[1], color=["#4C72B0", "#DD8452"], rot=15)
    axes[1].set_title("Distribución dorada vs predicha"); axes[1].set_ylabel("# claims")

    merged["correct"] = (merged["gold_label"] == merged["pred_label"])
    sns.histplot(data=merged, x="confidence", hue="correct", bins=15, multiple="stack",
                 palette={True: "#55A868", False: "#C44E52"}, ax=axes[2])
    axes[2].set_title("Confianza: aciertos vs errores")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "clasificacion.png", dpi=150)
    plt.show()

    # Calibración (ECE, 10 bins)
    bins = np.linspace(0, 1, 11)
    merged["bin"] = pd.cut(merged["confidence"].clip(0, 1), bins, include_lowest=True)
    cal = merged.groupby("bin", observed=True).agg(conf=("confidence", "mean"),
                                                   acc=("correct", "mean"), n=("correct", "size"))
    ece = float((cal["n"] / cal["n"].sum() * (cal["conf"] - cal["acc"]).abs()).sum())
    clf_results["ECE"] = ece
    plt.figure(figsize=(5, 5))
    plt.plot([0, 1], [0, 1], "k--", label="calibración perfecta")
    plt.plot(cal["conf"], cal["acc"], "o-", label=f"modelo (ECE = {ece:.3f})")
    plt.xlabel("Confianza media"); plt.ylabel("Exactitud"); plt.title("Diagrama de calibración")
    plt.legend(); plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "calibracion.png", dpi=150)
    plt.show()

### 11. Resumen final

In [ ]:
summary = {
    "n_queries_retrieval": int(len(qrels)),
    "corpus_docs": int(len(wiki_df)), "chunks": int(len(chunks)),
    "retrieval": mean_metrics[["nDCG@10", "Recall@10", "Precision@10", "MRR@10", "Hit@10"]].round(4).to_dict(orient="index"),
    "best_alpha_ndcg10": float(best_a),
    "citation_coverage": round(float(coverage), 4),
    "citation_precision_vs_qrels": None if cite_ok != cite_ok else round(float(cite_ok), 4),
    "classification": {k: (round(float(v), 4) if isinstance(v, (int, float, np.floating)) else v)
                       for k, v in clf_results.items()} or "sin etiquetas doradas",
}
with open(OUTPUT_DIR / "resumen.json", "w", encoding="utf-8") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)
print(json.dumps(summary, indent=2, ensure_ascii=False))
print("\nArchivos generados en", OUTPUT_DIR.resolve(), ":", sorted(p.name for p in OUTPUT_DIR.iterdir()))

# Implementación gráfica

In [ ]:
!pip install gradio plotly pandas

Ejemplos:

Heaven Can Wait is a film adaptation of Harry Segall's play "Hamlet".

 Golf is played on a course.

 Louis Malle produced films in the French language.

In [ ]:
import gradio as gr
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def pipeline_verificacion(claim):
    # 1. Obtener scores y realizar el Retrieval real para el claim ingresado
    try:
        # Calculamos los scores léxicos y densos usando las funciones definidas
        sparse_score, dense_score = score_all(claim)
        retrieved = top_chunks(sparse_score, dense_score, RETRIEVAL_MODE, K)

        # 2. Inferencia NLI real para el claim y sus evidencias recuperadas
        probs = nli_probs(claim, retrieved) if VERIFIER_MODE in ("nli", "hybrid") else None
        label, state, conf, ci = classify_claim(retrieved, probs)

        # Extraer el chunk de soporte citado
        cited = retrieved.iloc[ci]
        chunk_de_soporte = f"[{cited['title']}] {cited['text']}"
        texto_confianza = f"{conf:.2%}"
    except Exception as e:
        return "ERROR", "0.00%", f"Ocurrió un error en el pipeline real: {str(e)}", go.Figure()

    # 3. Gráfica real con las métricas del modo híbrido calculadas previamente
    fig = go.Figure()
    k_values = EVAL_KS

    # Extraemos Recall y Precision para el modo híbrido configurado
    recall_scores = [mean_metrics.loc["hybrid", f"Recall@{k}"] for k in EVAL_KS]
    precision_scores = [mean_metrics.loc["hybrid", f"Precision@{k}"] for k in EVAL_KS]

    fig.add_trace(go.Scatter(x=k_values, y=recall_scores, mode='lines+markers', name='Recall@k', line=dict(color='royalblue')))
    fig.add_trace(go.Scatter(x=k_values, y=precision_scores, mode='lines+markers', name='Precision@k', line=dict(color='firebrick')))
    fig.update_layout(
        title="Rendimiento de Recuperación Global (Modo Híbrido)",
        xaxis=dict(title="Top K", tickmode='linear', tick0=1, dtick=1),
        yaxis=dict(title="Score", range=[0, 1.05]),
        margin=dict(l=40, r=40, t=40, b=40)
    )

    return label, texto_confianza, chunk_de_soporte, fig

# ==========================================
# INTERFAZ GRÁFICA EN COLAB (GRADIO)
# ==========================================
with gr.Blocks() as interfaz:
    gr.Markdown("# Verificador de Hechos RAG (Dataset FEVER) - Pipeline Real")
    gr.Markdown("Esta interfaz ejecuta la recuperación híbrida real y clasifica la afirmación usando el modelo NLI cargado.")

    with gr.Row():
        claim_input = gr.Textbox(
            label="Afirmación (Claim)",
            placeholder="Escribe la afirmación aquí... p.ej. Golf is played on a course.",
            scale=4
        )
        btn_verificar = gr.Button("Verificar", variant="primary", scale=1)

    with gr.Row():
        estado_output = gr.Textbox(label="Estado (Predicción FEVER)")
        confianza_output = gr.Textbox(label="Confianza Combinada")

    evidencia_output = gr.Textbox(
        label="Evidencia Citada (Mejor Fragmento)",
        lines=3
    )

    grafica_output = gr.Plot(label="Métricas de Recuperación")

    btn_verificar.click(
        fn=pipeline_verificacion,
        inputs=claim_input,
        outputs=[estado_output, confianza_output, evidencia_output, grafica_output]
    )

# Lanzar la aplicación
interfaz.launch(debug=True, inline=True)

# Bibliografía



1. **FEVER: a large-scale dataset for fact extraction and VERification**  
   Thorne et al., 2018.  
   Dataset y descripción de la tarea de verificación de afirmaciones.  
   https://arxiv.org/abs/1803.05355  

2. **BEIR: A Heterogeneous Benchmark for Zero-shot Evaluation of Information Retrieval Models**  
   Thakur et al., 2021.  
   Benchmark que incluye FEVER y define métricas como nDCG@10, MRR, Recall@k.  
   https://arxiv.org/abs/2104.08663  
   https://github.com/beir-cellar/beir  

3. **Sentence Transformers documentation**  
   Documentación oficial de `sentence-transformers`, modelos para embeddings de frases.  
   https://www.sbert.net/  
   https://huggingface.co/sentence-transformers/all-MiniLM-L6-v2  

4. **FAISS: A Library for Efficient Similarity Search**  
   Facebook AI Similarity Search (FAISS).  
   https://github.com/facebookresearch/faiss  

5. **Rank BM25**  
   Implementación en Python del algoritmo BM25 para ranking de documentos.  
   https://github.com/dorianbrown/rank_bm25  

6. **Hugging Face Transformers – NLI models**  
   Documentación y ejemplos de modelos de inferencia de lenguaje natural (NLI).  
   https://huggingface.co/docs/transformers  
   Ejemplo de modelo NLI: https://huggingface.co/cross-encoder/nli-deberta-v3-small  

7. **Evidence-based Interpretable Open-domain Fact-checking**  
   Trabajo que evalúa sistemas de fact‑checking sobre FEVER con evidencia explícita.  
   https://ar5iv.org/html/2312.05834  